# RecSys-AI — Train trên Colab (GPU T4)

Notebook này chạy **toàn bộ training trên Colab**, KHÔNG train local.
Repo: `https://github.com/imtarget05/ai-product-recommender.git`

## Các bước

1. Colab menu **Runtime → Change runtime type → T4 GPU** (miễn phí).
2. Chạy lần lượt từng cell từ trên xuống dưới (Shift+Enter).
3. Mọi path đều là `/content/ai-product-recommender/...` — không dùng path máy local.
4. Cell cuối copy artifact về Google Drive; kéo về máy local theo hướng dẫn ở cell cuối cùng.

## Pipeline trong notebook (map với source repo)

- Seed DB: `scripts/generate_seed_data.py` (gọi `init_db()` + sinh catalog/users/interactions).
- Train: `HybridRecommender.fit(products, interactions)` — xem `src/models/hybrid.py`; logic batch y hệt `scripts/retrain_worker.py`.
- Evaluate: `scripts/evaluate_models.py` → P@K / R@K / NDCG@K + Coverage (temporal split chống leakage).
- Artifact: `data/embeddings/item_embeddings.npy` + `item_embedder_meta.pkl` (xem `src/features/text_embedder.py::ItemEmbedder.save`) + snapshot tar.gz (kèm Qdrant snapshot nếu có).

In [ ]:
# Cell 1 — Kiểm tra GPU (expect: Tesla T4)
!nvidia-smi

try:
    import torch
    print('torch:', torch.__version__, '| cuda_available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except ImportError:
    print('torch chưa cài (không bắt buộc — model hiện tại là TF-IDF/SVD + CF latent, chạy CPU được).')

Thu Sep 17 18:56:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Cell 2 — Clone repo (thay <REPO_URL> nếu fork riêng)
# NOTE: repo GitHub tên ai-product-recommender, folder local tên RecSys-AI
# (cùng remote https://github.com/imtarget05/ai-product-recommender.git —
# `git remote -v` trong ~/Downloads/RecSys-AI trỏ về ai-product-recommender,
# folder local RecSys-AI chỉ là tên thư mục trên máy, không phải tên repo GitHub).
# Trên Colab folder clone là /content/ai-product-recommender (theo tên repo GitHub).
REPO_URL = 'https://github.com/imtarget05/ai-product-recommender.git'
REPO_DIR = '/content/ai-product-recommender'

import os
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
    # Chuẩn hoá: nếu ai đó clone mặc định (không truyền $REPO_DIR) thì mv về đúng thư mục
    !test -d /content/ai-product-recommender || (test -d /content/RecSys-AI && mv /content/RecSys-AI /content/ai-product-recommender) || true
else:
    print('Repo đã tồn tại, pull mới nhất...')
    !git -C $REPO_DIR pull --ff-only

%cd /content/ai-product-recommender
!git log --oneline -3
!ls


Repo đã tồn tại, pull mới nhất...
Already up to date.
/content/ai-product-recommender
6e06f55 (HEAD -> main, origin/main, origin/HEAD) feat: add system verification documentation, enhance cache management, improve RAG search availability checks, and implement regression tests
bb83002 feat(agent): implement RAG semantic search and conversational shopping assistant
be3964f Add technical design specification for RAG and shopping assistant
alembic      data    docs      pytest.ini    README.md	       scripts	tests
alembic.ini  docker  Procfile  railway.json  requirements.txt  src


In [ ]:
# Cell 3 — Cài dependencies
%cd /content/ai-product-recommender
# Cài đặt requirements nhưng bỏ qua việc downgrade làm gãy numpy, hoặc ép buộc cập nhật lại các thư viện liên quan để đồng bộ binary
!pip install -q -r requirements.txt
!pip install -q -U numpy scipy scikit-learn
print('✅ pip install và đồng bộ numpy/scipy/scikit-learn xong.')

/content/ai-product-recommender
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 68.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.8.23 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
dm-tree 0.1.10 requires numpy>=2.1.0; python_version >= "3.13", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.14.0.94 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.

In [ ]:
# Cell 4 — Prepare / Seed database (scripts/generate_seed_data.py)
# Thêm PYTHONPATH để Python nhận diện được module 'src'
%cd /content/ai-product-recommender
!ls scripts/
!PYTHONPATH=. python scripts/generate_seed_data.py
!ls -la data/

/content/ai-product-recommender
download_dataset.py  generate_seed_data.py   retrain_worker.py
evaluate_models.py   ingest_retailrocket.py
🌱 Initializing Database Schema...
Database already contains data. Clearing old seed data for fresh generation...
📦 Seeding 31 Products...
/content/ai-product-recommender/scripts/generate_seed_data.py:85: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at=datetime.utcnow() - timedelta(days=random.randint(10, 180))
👥 Generating 150 Users with behavioral personas...
/content/ai-product-recommender/scripts/generate_seed_data.py:120: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at=datetime.utcnow() - timedelta(days=random.randi

In [ ]:
# Cell 5 — Train HybridRecommender + save embedder (mirror scripts/retrain_worker.py)
%cd /content/ai-product-recommender

import sys
import os
if '/content/ai-product-recommender' not in sys.path:
    sys.path.append('/content/ai-product-recommender')

from src.database.session import get_db_context
from src.database.models import Product, Interaction
from src.models.hybrid import HybridRecommender
from src.config import settings

with get_db_context() as db:
    products = db.query(Product).all()
    interactions = db.query(Interaction).all()
    print(f'📦 Loaded {len(products)} products, {len(interactions)} interactions.')
    assert products, 'DB trống — chạy lại Cell 4 (generate_seed_data).'

    hybrid = HybridRecommender(
        cf_weight=settings.HYBRID_CF_WEIGHT,      # 0.6
        cb_weight=settings.HYBRID_CB_WEIGHT,      # 0.4
        cold_start_threshold=settings.COLD_START_THRESHOLD,  # 3
    )
    print('🧠 Fitting sub-models (Popularity, Content-Based, Latent CF)...')
    hybrid.fit(products, interactions)

    print(f'💾 Saving embeddings to {settings.EMBEDDINGS_DIR}...')
    hybrid.content_model.embedder.save(settings.EMBEDDINGS_DIR)

print('✅ Train xong. Artifact:')
!ls -la /content/ai-product-recommender/data/embeddings/

/content/ai-product-recommender
📦 Loaded 31 products, 4310 interactions.
🧠 Fitting sub-models (Popularity, Content-Based, Latent CF)...
Fitting Popularity Baseline...
Fitting Content-Based Model...
Fitting Collaborative Filtering Model...
✅ Hybrid Recommender successfully trained all sub-models.
💾 Saving embeddings to /content/ai-product-recommender/data/embeddings...
✅ Train xong. Artifact:
total 304
drwxr-xr-x 2 root root   4096 Sep 17 18:57 .
drwxr-xr-x 5 root root   4096 Sep 17 18:57 ..
-rw-r--r-- 1 root root      0 Sep 17 18:40 .gitkeep
-rw-r--r-- 1 root root 285733 Sep 17 18:57 item_embedder_meta.pkl
-rw-r--r-- 1 root root  16000 Sep 17 18:57 item_embeddings.npy


In [ ]:
# Cell 6 — Evaluate: P@K / R@K / NDCG@K + Coverage (strict temporal split)
%cd /content/ai-product-recommender
!python scripts/evaluate_models.py --top-k 5 --output reports/eval_colab_t4_top5.json
!python scripts/evaluate_models.py --top-k 10 --output reports/eval_colab_t4_top10.json
!ls -la reports/

/content/ai-product-recommender
/content/ai-product-recommender/src/evaluation/metrics.py:73: SyntaxWarning: invalid escape sequence '\s'
  DCG = \sum_{i=1}^k (2^{rel_i} - 1) / log2(i + 1)
🎯 RECSYS-AI MODEL BENCHMARK & OFFLINE EVALUATION (STRICT TEMPORAL)
Loaded 31 catalog products and 4310 interactions.
Global Cutoff Timestamp (T_cutoff): 2026-09-11 16:07:33.086997
Split completed: 3449 train interactions, 126 active test users.
----------------------------------------------------------------------
⚡ Training & evaluating: PopularityBaseline...
⚡ Training & evaluating: ContentBased...
⚡ Training & evaluating: CollaborativeFiltering...
⚡ Training & evaluating: HybridRecommender...
Fitting Popularity Baseline...
Fitting Content-Based Model...
Fitting Collaborative Filtering Model...
✅ Hybrid Recommender successfully trained all sub-models.
⚡ Evaluating Full Pipeline: Hybrid + Multi-Objective Reranker...

Model / Pipeline               | P@5      | R@5      | NDCG@5   | Coverage
--------

In [ ]:
# Cell 7 — Đóng gói snapshot (embedder + eval reports + Qdrant snapshot nếu có)
%cd /content/ai-product-recommender
import datetime
STAMP = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
SNAP = f'/content/recsys_artifacts_{STAMP}.tar.gz'

# Qdrant snapshot: nếu dùng Qdrant local/cloud thì export trước tại đây
# (mặc định train dùng local embeddings, không bắt buộc Qdrant).
# Ví dụ Qdrant local: curl -X POST http://localhost:6333/snapshots

!mkdir -p reports
!tar -czf $SNAP -C /content/ai-product-recommender data/embeddings reports
!ls -lh $SNAP
print(f'SNAPSHOT={SNAP}')

/content/ai-product-recommender


/tmp/ipykernel_10126/3649044336.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  STAMP = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')


-rw-r--r-- 1 root root 98K Sep 17 18:57 /content/recsys_artifacts_20260917_185746.tar.gz
SNAPSHOT=/content/recsys_artifacts_20260917_185746.tar.gz


In [ ]:
# Cell 8 — Copy artifact về Google Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/recsys_artifacts
!cp /content/recsys_artifacts_*.tar.gz /content/drive/MyDrive/recsys_artifacts/
!cp /content/ai-product-recommender/reports/eval_colab_t4_*.json /content/drive/MyDrive/recsys_artifacts/
!ls -lh /content/drive/MyDrive/recsys_artifacts/

Mounted at /content/drive
cp: cannot stat '/content/ai-product-recommender/reports/eval_colab_t4_*.json': No such file or directory
total 98K
-rw------- 1 root root 98K Sep 17 18:58 recsys_artifacts_20260917_185746.tar.gz


In [ ]:
# Cell 9 — Gom toàn bộ file cần thiết xuống Local
import os
import shutil
from google.colab import files

# Tạo thư mục tạm để gom các file cần thiết
export_dir = '/content/recsys_local_package'
os.makedirs(export_dir, exist_ok=True)
os.makedirs(f'{export_dir}/embeddings', exist_ok=True)
os.makedirs(f'{export_dir}/reports', exist_ok=True)
os.makedirs(f'{export_dir}/database', exist_ok=True)

# 1. Sao chép model embeddings
src_embed_dir = '/content/ai-product-recommender/data/embeddings'
if os.path.exists(src_embed_dir):
    for file in os.listdir(src_embed_dir):
        if file.endswith(('.npy', '.pkl')):
            shutil.copy(os.path.join(src_embed_dir, file), f'{export_dir}/embeddings/')

# 2. Sao chép database sqlite hiện tại
src_db = '/content/ai-product-recommender/data/recsys.db'
if os.path.exists(src_db):
    shutil.copy(src_db, f'{export_dir}/database/')

# 3. Sao chép các báo cáo đánh giá nếu có
src_reports_dir = '/content/ai-product-recommender/reports'
if os.path.exists(src_reports_dir):
    for file in os.listdir(src_reports_dir):
        if file.endswith('.json'):
            shutil.copy(os.path.join(src_reports_dir, file), f'{export_dir}/reports/')

# 4. Nén tất cả thành 1 file duy nhất
zip_path = '/content/recsys_local_package_full'
shutil.make_archive(zip_path, 'zip', export_dir)
zip_file = f'{zip_path}.zip'

# 5. Sao lưu thêm một bản sang Google Drive
drive_dest_dir = '/content/drive/MyDrive/recsys_artifacts'
os.makedirs(drive_dest_dir, exist_ok=True)
shutil.copy(zip_file, drive_dest_dir)
print(f'✅ Đã lưu bản backup vào Google Drive: {drive_dest_dir}/recsys_local_package_full.zip')

# 6. Kích hoạt tải trực tiếp xuống máy local qua trình duyệt
print('📥 Đang chuẩn bị tải trực tiếp file zip xuống máy...')
files.download(zip_file)

✅ Đã lưu bản backup vào Google Drive: /content/drive/MyDrive/recsys_artifacts/recsys_local_package_full.zip
📥 Đang chuẩn bị tải trực tiếp file zip xuống máy...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Kéo artifact về máy local

### Cách A — Google Drive (khuyên dùng)

Mở Drive → thư mục `recsys_artifacts/` → tải file `recsys_artifacts_<STAMP>.tar.gz` về máy, rồi:

```bash
cd ~/Downloads/RecSys-AI   # thư mục repo trên máy bạn
tar -xzf ~/Downloads/recsys_artifacts_<STAMP>.tar.gz -C /tmp/recsys_art
cp /tmp/recsys_art/data/embeddings/item_embeddings.npy ./data/embeddings/
cp /tmp/recsys_art/data/embeddings/item_embedder_meta.pkl ./data/embeddings/
cp /tmp/recsys_art/reports/eval_colab_t4_*.json ./reports/
```

### Cách B — Tải trực tiếp từ Colab

```python
from google.colab import files
files.download('/content/recsys_artifacts_<STAMP>.tar.gz')  # thay STAMP thực tế ở Cell 7
```

### Chạy serving API với model mới (trên máy local)

API tự load embedder từ `data/embeddings/` khi khởi động; hoặc trigger hot-reload không downtime (xem `scripts/retrain_worker.py`):

```bash
curl -X POST http://127.0.0.1:8000/api/v1/admin/reload-model
```